# 🕵️ Dia 52 — Artefatos NTFS: a MFT e a manipulação de timestamps

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 52* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-52.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A MFT do `SA-CX-07` diz que o `msa-atualizador.exe` nasceu em 2021. A pasta dele nasceu em 5 de outubro de 2026. Rafa: *"Uma dessas duas mentiu."*

Neste notebook você vai:

1. **Ler o trecho da MFT do WS-FIN-07** (LAB-P4) e refazer o `csvcut`/`csvgrep` com o módulo `csv`.
2. **Calcular os sinais de *timestomping*** automaticamente, um por coluna.
3. **Gerar a tabela de anomalias** e a mini linha do tempo.
4. **Repetir com a MFT do `SA-CX-07`**, onde há três acertos e nem todos são culpados.
5. **Jogar o quiz da explicação inocente.**

🐍 Só biblioteca padrão. Offline-first.


In [ ]:
# ------------------------------------------------------------
# RECEBER O TRECHO DA MFT (EVID-002)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/filesystem/ws-fin-07-mft-excerpt.csv'] = ''.join((
    'EntryNumber,SequenceNumber,InUse,ParentEntryNumber,ParentPath,FileName,Extension,FileSize,IsDirectory,SI<FN,uSecZeros,Created0x10,Created0x30,LastModified0x10,LastModified0x30,LastRecordChange0x10,LastRecordChange0x30,LastAccess0x10,LastAccess0x30\n'
    '88412,3,True,88390,.\\Users\\acct.clerk01\\AppData\\Roaming,SyncHelper,,0,True,False,False,2026-03-13 16:42:29.5528113,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113\n'
    '88413,1,True,88412,.\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper,synchelper.exe,.exe,412160,False,True,True,2019-06-11 09:14:00.0000000,2026-03-13 16:42:30.1102964,2019-06-11 09:14:00.0000000,2026-03-13 16:42:30.1102964,2026-03-13 16:42:31.0470021,2026-03-13 16:42:30.1102964,2026-03-13 16:42:31.0470021,2026-03-13 16:42:30.1102964\n'
    '88414,1,True,88412,.\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper,config.dat,.dat,1896,False,False,False,2026-03-13 16:42:30.2231870,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870\n'
    '88420,2,True,1471,.\\Users\\Public,svchost.exe,.exe,98304,False,True,True,2018-09-15 07:28:00.0000000,2026-03-13 16:44:08.8811220,2018-09-15 07:28:00.0000000,2026-03-13 16:44:08.8811220,2026-03-13 16:44:09.0034187,2026-03-13 16:44:08.8811220,2026-03-13 16:44:09.0034187,2026-03-13 16:44:08.8811220\n'
    '23105,7,True,23100,.\\Windows\\System32,svchost.exe,.exe,57528,False,False,False,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4562810,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4562810,2025-11-12 03:41:17.4406211,2026-03-13 12:58:04.1101114,2025-11-12 03:41:17.4406211\n'
    '77310,4,True,77290,.\\Users\\acct.clerk01\\Documents\\Invoices,inv-1438.pdf,.pdf,184233,False,False,False,2026-03-11 15:02:44.9021133,2026-03-11 15:02:44.9021133,2026-03-11 15:02:45.0019224,2026-03-11 15:02:44.9021133,2026-03-11 15:02:45.0019224,2026-03-11 15:02:44.9021133,2026-03-13 14:20:31.7710422,2026-03-11 15:02:44.9021133\n'
    '88398,2,False,88396,.\\Users\\acct.clerk01\\AppData\\Local\\Microsoft\\Windows\\INetCache\\Content.Outlook\\K3Q9ZP1L,Invoice_0313.zip,.zip,210944,False,False,False,2026-03-13 16:41:52.6612004,2026-03-13 16:41:52.6612004,2026-03-13 16:41:52.7120391,2026-03-13 16:41:52.6612004,2026-03-13 16:48:03.1177420,2026-03-13 16:41:52.6612004,2026-03-13 16:41:55.0210933,2026-03-13 16:41:52.6612004\n'
    '61022,11,True,61000,.\\Users\\acct.clerk01\\Desktop,notes.txt,.txt,3120,False,False,False,2026-02-20 14:10:05.1203311,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 1. `csvcut ... | csvlook`

Uma tabela legível com as colunas que importam. A função `csvlook()` abaixo imita o `csvlook` do csvkit.


In [ ]:
import csv, io

def ler_csv(texto):
    """Lê o CSV do MFTECmd numa lista de dicionários (uma linha = um registro da MFT)."""
    return list(csv.DictReader(io.StringIO(texto)))

def csvlook(linhas, colunas):
    """Imprime as colunas pedidas alinhadas, como o csvlook."""
    larg = {c: max(len(c), *(len(str(l[c])) for l in linhas)) for c in colunas}
    print("| " + " | ".join(c.ljust(larg[c]) for c in colunas) + " |")
    print("|" + "|".join("-" * (larg[c] + 2) for c in colunas) + "|")
    for l in linhas:
        print("| " + " | ".join(str(l[c]).ljust(larg[c]) for c in colunas) + " |")

MFT = ler_csv(EVIDENCIA["ws-fin-07-mft-excerpt.csv"])
csvlook(MFT, ["EntryNumber", "InUse", "FileName", "SI<FN", "uSecZeros", "Created0x10", "Created0x30"])


## 2. `csvgrep -c 'SI<FN' -m True`


In [ ]:
flagados = [l for l in MFT if l["SI<FN"] == "True"]
csvlook(flagados, ["FileName", "ParentPath", "Created0x10", "Created0x30", "uSecZeros"])
print(f"\n{len(flagados)} linhas (o original espera 2)")


## 3. Os sinais, um por coluna

Para cada arquivo, calculamos:

| Sinal | Como |
|---|---|
| `SI<FN` | `Created0x10 < Created0x30` (a *flag* do MFTECmd) |
| `uSecZeros` | os 7 dígitos depois do segundo são `0000000` |
| mais velho que a pasta | `Created0x10` do arquivo < `Created0x30` da pasta-mãe (pelo `ParentEntryNumber`) |
| `$SI` reescrito logo depois | `LastRecordChange0x10` até 5 s depois do `Created0x30` |
| disfarce | nome de binário do sistema fora de `Windows\System32` |

Nenhum sinal é prova sozinho. A função devolve a **lista**, e você decide.


In [ ]:
from datetime import datetime

def t(s):
    """'2026-03-13 16:42:30.1102964' -> datetime (corta para microssegundos, que o datetime aceita)."""
    base, _, frac = s.partition(".")
    return datetime.strptime(base, "%Y-%m-%d %H:%M:%S").replace(microsecond=int((frac + "000000")[:6]))

NOMES_DE_SISTEMA = {"svchost.exe", "lsass.exe", "csrss.exe", "services.exe", "explorer.exe", "rundll32.exe"}

def sinais(linha, por_entrada):
    s = []
    if t(linha["Created0x10"]) < t(linha["Created0x30"]):
        s.append("SI<FN")
    if linha["Created0x10"].endswith(".0000000"):
        s.append("uSecZeros")
    mae = por_entrada.get(linha["ParentEntryNumber"])
    if mae and t(linha["Created0x10"]) < t(mae["Created0x30"]):
        dif = (t(mae["Created0x30"]) - t(linha["Created0x30"])).total_seconds()
        s.append(f"mais velho que a pasta-mãe {mae['FileName']} ({mae['EntryNumber']}, criada {mae['Created0x30'][:19]})")
    rc = (t(linha["LastRecordChange0x10"]) - t(linha["Created0x30"])).total_seconds()
    if "SI<FN" in s and 0 <= rc <= 5:
        s.append(f"LastRecordChange0x10 {linha['LastRecordChange0x10'][11:19]}, {rc:.1f}s após o FN Created")
    if linha["FileName"].lower() in NOMES_DE_SISTEMA and "System32" not in linha["ParentPath"]:
        s.append(f"nome de sistema fora do System32 ({linha['ParentPath']})")
    return s

POR_ENTRADA = {l["EntryNumber"]: l for l in MFT}
for l in MFT:
    s = sinais(l, POR_ENTRADA)
    marca = "🔴" if "SI<FN" in s else ("⚪" if l["InUse"] == "False" else "  ")
    print(f"{marca} {l['EntryNumber']:>6} {l['FileName']:22} {'; '.join(s) or '-'}")


> 🔎 Repare no `svchost.exe` 23105 (System32): nenhum sinal. Ele é o **controle**, o arquivo legítimo com que você compara.

## 4. A linha apagada: `Invoice_0313.zip`

O que você pode dizer, em números:


In [ ]:
zipe = next(l for l in MFT if l["FileName"] == "Invoice_0313.zip")
pasta = next(l for l in MFT if l["FileName"] == "SyncHelper")
exe = next(l for l in MFT if l["EntryNumber"] == "88413")
# Contamos em segundos inteiros (como os horários aparecem na tabela): 16:41:52 -> 16:42:30 = 38 s
seg = lambda s: datetime.strptime(s[:19], "%Y-%m-%d %H:%M:%S")
gap = (seg(exe["Created0x30"]) - seg(zipe["Created0x30"])).total_seconds()
print(f"InUse={zipe['InUse']}  pasta: {zipe['ParentPath']}")
print(f"criado {zipe['Created0x30']}, registro alterado {zipe['LastRecordChange0x10']}")
print(f"O synchelper.exe ($FN) aparece {gap:.0f} s depois; a pasta SyncHelper, {(seg(pasta['Created0x30']) - seg(zipe['Created0x30'])).total_seconds():.0f} s depois.")
print("\nFrase calibrada: 'Um zip chamado Invoice_0313.zip existiu no cache do Outlook e PRECEDEU o",
      f"synchelper.exe em {gap:.0f} s.' O registro NÃO mostra que o zip continha o synchelper.exe.")


## 5. A tabela de anomalias (artefato)

Preencha as alternativas e a confiança pelo formulário. Os sinais já vêm calculados.


In [ ]:
#@title Sua análise do LAB-P4 { display-mode: "form" }
alt_synchelper = "Instalador legítimo gravando data de build: descartada, porque a pasta-mãe foi criada 1 s antes e o pai do processo é o PowerShell oculto (Dia 53)"  #@param {type:"string"}
conf_synchelper = "altamente provável"  #@param ["confirmado", "altamente provável", "provável", "possível"]
alt_svchost = "Cópia entre volumes de um svchost legítimo: improvável, um svchost legítimo não vive em Users\\Public"  #@param {type:"string"}
conf_svchost = "provável"  #@param ["confirmado", "altamente provável", "provável", "possível"]

from pathlib import Path
ELEVARIA = "$UsnJrnl com FileCreate e depois BasicInfoChange na mesma entrada em segundos; Prefetch com a 1ª execução"
linhas_md = ["| File | MFT entry | Observed signals | Alternatives considered | Confidence | Evidence that would raise it |",
             "|---|---|---|---|---|---|"]
for l, alt, conf in [(next(x for x in MFT if x["EntryNumber"] == "88413"), alt_synchelper, conf_synchelper),
                     (next(x for x in MFT if x["EntryNumber"] == "88420"), alt_svchost, conf_svchost)]:
    nome = l["FileName"] + (" (Users\\Public)" if "Public" in l["ParentPath"] else "")
    linhas_md.append(f"| {nome} | {l['EntryNumber']} | {'; '.join(sinais(l, POR_ENTRADA))} | {alt} | {conf} | {ELEVARIA} |")

mini = ["", "## Mini linha do tempo, 13/mar/2026 UTC", "",
        "- 16:41:52 Invoice_0313.zip criado no cache do Outlook (88398, depois apagado)",
        "- 16:42:29 pasta SyncHelper criada (88412)",
        "- 16:42:30 synchelper.exe criado segundo o $FN (88413); $SI diz 2019; registro alterado 16:42:31",
        "- 16:44:08 svchost.exe criado em Users\\Public segundo o $FN (88420); $SI diz 2018"]
texto = "# Day 52 timestamp anomalies (LAB-P4)\n\n" + "\n".join(linhas_md + mini) + "\n"
Path("notes").mkdir(exist_ok=True)
Path("notes/day52-timestamp-anomalies.md").write_text(texto, encoding="utf-8")
print(texto)


## 6. ➕ A MFT do `SA-CX-07`


In [ ]:
# ------------------------------------------------------------
# RECEBER A MFT DO SA-CX-07
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-52/sa-cx-07-mft-excerpt.csv'] = ''.join((
    'EntryNumber,SequenceNumber,InUse,ParentEntryNumber,ParentPath,FileName,Extension,FileSize,IsDirectory,SI<FN,uSecZeros,Created0x10,Created0x30,LastModified0x10,LastModified0x30,LastRecordChange0x10,LastRecordChange0x30,LastAccess0x10,LastAccess0x30\n'
    '120931,2,True,98112,.\\Users\\cx.centro07\\Downloads,Modulo_Seguranca_SerraAzul.zip,.zip,410624,False,False,False,2026-10-05 11:23:14.2210931,2026-10-05 11:23:14.2210931,2026-10-05 11:23:14.2210931,2026-10-05 11:23:14.2210931,2026-10-05 11:23:15.0102215,2026-10-05 11:23:14.2210931,2026-10-05 11:24:27.7700412,2026-10-05 11:23:14.2210931\n'
    '120944,1,True,98090,.\\Users\\cx.centro07\\AppData\\Roaming,MSA,,0,True,False,False,2026-10-05 11:24:28.9012211,2026-10-05 11:24:28.9012211,2026-10-05 11:24:29.6001234,2026-10-05 11:24:28.9012211,2026-10-05 11:24:29.6001234,2026-10-05 11:24:28.9012211,2026-10-05 11:24:28.9012211,2026-10-05 11:24:28.9012211\n'
    '120945,1,True,120944,.\\Users\\cx.centro07\\AppData\\Roaming\\MSA,msa-atualizador.exe,.exe,389120,False,True,True,2021-02-03 10:00:00.0000000,2026-10-05 11:24:29.4410087,2021-02-03 10:00:00.0000000,2026-10-05 11:24:29.4410087,2026-10-05 11:24:30.1180342,2026-10-05 11:24:29.4410087,2021-02-03 10:00:00.0000000,2026-10-05 11:24:29.4410087\n'
    '120946,1,True,120944,.\\Users\\cx.centro07\\AppData\\Roaming\\MSA,config.ini,.ini,744,False,False,False,2026-10-05 11:24:29.6001234,2026-10-05 11:24:29.6001234,2026-10-05 11:24:30.2051177,2026-10-05 11:24:29.6001234,2026-10-05 11:24:30.2051177,2026-10-05 11:24:29.6001234,2026-10-05 11:24:29.6001234,2026-10-05 11:24:29.6001234\n'
    '118020,1,True,98100,.\\Users\\cx.centro07\\Documents,Normativos,,0,True,False,False,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870,2026-10-02 14:11:06.5521870\n'
    '118021,1,True,118020,.\\Users\\cx.centro07\\Documents\\Normativos,tabela-tarifas-2026.xlsx,.xlsx,88230,False,True,True,2025-12-18 08:00:00.0000000,2026-10-02 14:11:06.6013377,2025-12-18 08:00:00.0000000,2026-10-02 14:11:06.6013377,2026-10-02 14:11:06.6201145,2026-10-02 14:11:06.6013377,2025-12-18 08:00:00.0000000,2026-10-02 14:11:06.6013377\n'
    '118022,1,True,118020,.\\Users\\cx.centro07\\Documents\\Normativos,manual-caixa-v7.pdf,.pdf,1402220,False,True,True,2025-11-30 17:42:00.0000000,2026-10-02 14:11:06.6390021,2025-11-30 17:42:00.0000000,2026-10-02 14:11:06.6390021,2026-10-02 14:11:06.6512003,2026-10-02 14:11:06.6390021,2025-11-30 17:42:00.0000000,2026-10-02 14:11:06.6390021\n'
    '118019,3,False,98112,.\\Users\\cx.centro07\\Downloads,pacote-normativos.zip,.zip,1350012,False,False,False,2026-10-02 14:10:58.1100224,2026-10-02 14:10:58.1100224,2026-10-02 14:10:58.1100224,2026-10-02 14:10:58.1100224,2026-10-02 14:12:40.0021110,2026-10-02 14:10:58.1100224,2026-10-02 14:10:58.1100224,2026-10-02 14:10:58.1100224\n'
    '23111,9,True,23100,.\\Windows\\System32,schtasks.exe,.exe,235008,False,False,False,2025-11-12 03:41:20.1180032,2025-11-12 03:41:20.1180032,2025-11-12 03:41:20.1180032,2025-11-12 03:41:20.1180032,2025-11-12 03:41:20.1180032,2025-11-12 03:41:20.1180032,2026-10-05 11:24:44.2001190,2025-11-12 03:41:20.1180032\n'
    '64020,2,True,64001,.\\Program Files\\SerraAzul\\Caixa,caixa-sa.exe,.exe,2048000,False,False,False,2026-08-14 19:20:33.4410221,2026-08-14 19:20:33.4410221,2026-08-14 19:20:33.4410221,2026-08-14 19:20:33.4410221,2026-08-14 19:20:33.4410221,2026-08-14 19:20:33.4410221,2026-10-05 10:55:01.0012334,2026-08-14 19:20:33.4410221\n'
))

MFT_SA_TXT = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    MFT_SA_TXT[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
MFT_SA = ler_csv(MFT_SA_TXT["sa-cx-07-mft-excerpt.csv"])
POR_ENTRADA_SA = {l["EntryNumber"]: l for l in MFT_SA}
print("csvgrep -c 'SI<FN' -m True:\n")
for l in MFT_SA:
    if l["SI<FN"] == "True":
        print(f"🔴 {l['EntryNumber']} {l['FileName']}")
        for s in sinais(l, POR_ENTRADA_SA):
            print(f"      - {s}")
print()
for nome in ("pacote-normativos.zip", "Modulo_Seguranca_SerraAzul.zip"):
    z = next(l for l in MFT_SA if l["FileName"] == nome)
    print(f"⚪ {z['EntryNumber']} {nome}: InUse={z['InUse']}, criado {z['Created0x30'][:19]} em {z['ParentPath']}")


> 🧠 **Três acertos, padrões iguais.** Os dois arquivos de `Normativos` têm exatamente os mesmos sinais do `msa-atualizador.exe`, e há um ZIP baixado oito segundos antes da pasta deles. A MFT sozinha **não separa** o inocente do malicioso. O que separa vem de fora: o proxy (Dia 49) com o `atualiza-modulo.example`, a memória (Dia 53), o `$UsnJrnl`. Se a sua confiança para o `msa-atualizador.exe` ficou em "altamente provável" só com a MFT, pergunte-se o que você usou além dela.

## 7. 🎮 Quiz da explicação inocente

Para cada caso, escolha a confiança que **só a evidência descrita** sustenta.


In [ ]:
MODO_JOGO = True
ESCALA = ["confirmado", "altamente provável", "provável", "possível"]

CASOS = [
    ("LAB-P4 synchelper.exe: SI<FN, uSecZeros, pasta-mãe 1 s antes, registro alterado 1 s depois, "
     "filho de PowerShell oculto (memória)", {2, 1},
     "Vários sinais independentes concordam e há fonte fora da MFT."),
    ("SA-CX-07 tabela-tarifas-2026.xlsx: SI<FN, uSecZeros, pasta criada 8 s após um ZIP de normativos baixado", {4},
     "Coerente com extrator de ZIP. Timestomp malicioso é só possível."),
    ("SA-CX-07 msa-atualizador.exe, SÓ com a MFT: SI<FN, uSecZeros, pasta-mãe criada 0,5 s antes, ZIP 74 s antes", {3, 4},
     "Os mesmos sinais dos Normativos. Sem outra fonte, não passa de provável."),
    ("Um único arquivo com SI<FN e mais nada (sem uSecZeros, pasta antiga)", {4},
     "Cópia entre volumes produz isso. Um sinal só não sustenta mais que 'possível'."),
]
acertos = 0
for i, (caso, aceitas, porque) in enumerate(CASOS, 1):
    print(f"\n[{i}] {caso}")
    for k, e in enumerate(ESCALA, 1):
        print(f"   {k}. {e}")
    if MODO_JOGO:
        r = input("   Confiança: ").strip()
        ok = r.isdigit() and int(r) in aceitas
        acertos += ok
        print("   ✅" if ok else "   ❌", end=" ")
    print("Aceitas:", " ou ".join(ESCALA[a - 1] for a in sorted(aceitas)), "—", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(CASOS)}")


In [ ]:
try:
    from google.colab import files
    files.download("notes/day52-timestamp-anomalies.md")
except ImportError:
    print("(Fora do Colab: notes/day52-timestamp-anomalies.md)")


## ✅ Antes de fechar o Dia 52

- [ ] tabela com os dois `SI<FN` do LAB-P4, cada um com 2+ sinais e uma alternativa
- [ ] "precedeu", nunca "continha", para `Invoice_0313.zip`
- [ ] ➕ tabela do `SA-CX-07` com **três** linhas e a explicação inocente dos Normativos
- [ ] em uma frase: por que `$FN` é mais difícil de alterar que `$SI` (sem dizer "impossível")

**Amanhã (Dia 53):** memória. Quem iniciou o `msa-atualizador.exe`? A árvore de processos responde.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
